In [7]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [ ]:
from transaction_analysis.data.loader import load_all_transactions, load_users
from transaction_analysis.stats.drift import drift_report

transactions = load_all_transactions()
users = load_users()

numeric_transaction = ["amount_usd"]
categorical_transaction = ["transaction_type", "mcc", "fraud"]

numeric_users = [
    "birth_date",
    "per_capita_income_usd",
    "yearly_income_usd",
    "total_debt_usd",
    "credit_score",
    "num_credit_cards",
]
categorical_users = ["gender"]

## Strategia wykrywania driftu

Metoda testu zależy od **typu zmiennej**:

| Typ zmiennej | Przykłady | Test | Miara siły efektu |
|---|---|---|---|
| numeryczna / porządkowa | `amount_usd`, `credit_score`, `birth_date` | Kołmogorow–Smirnow | KS_D + PSI |
| nominalna kategoryczna | `transaction_type`, `fraud`, `gender` | chi-kwadrat | Cramer's V + PSI |

Kołmogorow–Smirnow dla porównania wymaga **porządku** wartości - dlatego nie jest stosowany do zmiennych nominalnych

Dlaczego nie patrzymy na same p-wartości KS oraz Chi2? p-wartość zależy od **liczności** próby:<br>
Transakcje mają ~6,6 mln wierszy na połowę - przy takim `n` *każda* mała różnica powoduje `p -> 0`, więc p nie odróżnia driftu od szumu

Dlatego decyzję podejmujemy na podstawie PSI (Population Stability Index), który nie zależy od liczności próby:

| PSI | Interpretacja |
|---|---|
| `< 0.1` | znikomy drift |
| `0.1 – 0.25` | średni drift |
| `≥ 0.25` | duży drift |

Pomocniczo dla zmiennych kategorycznych - Cramer's V: 

| Cramer's V | Interpretacja |
|---|---|
| `< 0.1` | znikomy drift |
| `0.1 - 0.3` | mały drift |
| `0.3 – 0.5` | średni drift |
| `> 0.5` | duży drift |

In [9]:
transactions_old, transactions_new = (
    transactions[transactions["date"] < "2015-01-01"],
    transactions[transactions["date"] >= "2015-01-01"],
)
users_old, users_new = users[users["id"] < 1000], users[users["id"] >= 1000]
del transactions, users

In [10]:
transactions_new.head()

,transaction_id,date,client_id,card_id,amount_usd,transaction_type,merchant_id,merchant_city,merchant_state,zip,mcc,errors,fraud
6571667,15471192,2015-01-01 00:01:00,316,2038,69.550003,Chip Transaction,79360,Giddings,TX,78942,5411,NaN,<NA>
6571668,15471193,2015-01-01 00:01:00,1585,339,34.820000,Swipe Transaction,69972,Jacksonville,FL,32222,5814,NaN,False
6571669,15471194,2015-01-01 00:03:00,848,3915,64.400002,Chip Transaction,13051,Harwood,MD,20776,5813,NaN,False
6571670,15471195,2015-01-01 00:04:00,1797,300,47.930000,Chip Transaction,54343,San Leandro,CA,94577,4121,NaN,False
6571671,15471196,2015-01-01 00:05:00,1557,2471,25.750000,Online Transaction,9932,ONLINE,NaN,NaN,5311,NaN,False


In [11]:
drift_report(
    transactions_old,
    transactions_new,
    numeric_cols=numeric_transaction,
    categorical_cols=categorical_transaction,
)

,column,type,ks_stat,ks_pvalue,chi2_stat,chi2_pvalue,cramers_v,psi,verdict
0,amount_usd,numeric,0.005323,0.0,<NA>,<NA>,<NA>,0.000181,znikomy drift
1,transaction_type,categorical,<NA>,<NA>,7948464.656272,0.0,0.772893,10.758694,duży drift
2,mcc,categorical,<NA>,<NA>,3768.561136,0.0,0.016829,0.001135,znikomy drift
3,fraud,categorical,<NA>,<NA>,331.112967,0.0,0.006094,0.000150,znikomy drift


Jedyny rzeczywisty drift występuje w `transaction_type`

`amount_usd`, `mcc` oraz `fraud` są stabilne. Bardzo niskie p-wartości przy tych kolumnach to *artefakt ogromnego `n`*, a nie realny drift

In [12]:
drift_report(
    users_old,
    users_new,
    numeric_cols=numeric_users,
    categorical_cols=categorical_users,
)

,column,type,ks_stat,ks_pvalue,chi2_stat,chi2_pvalue,cramers_v,psi,verdict
0,birth_date,numeric,0.017,0.998723,<NA>,<NA>,<NA>,0.008000,znikomy drift
1,per_capita_income_usd,numeric,0.081,0.002818,<NA>,<NA>,<NA>,0.033420,znikomy drift
2,yearly_income_usd,numeric,0.055,0.097103,<NA>,<NA>,<NA>,0.021026,znikomy drift
3,total_debt_usd,numeric,0.065,0.029225,<NA>,<NA>,<NA>,0.031103,znikomy drift
4,credit_score,numeric,0.03,0.75937,<NA>,<NA>,<NA>,0.016560,znikomy drift
5,num_credit_cards,numeric,0.034,0.610166,<NA>,<NA>,<NA>,0.006514,znikomy drift
6,gender,categorical,<NA>,<NA>,0.002001,0.964325,0.001,0.000016,znikomy drift


Brak istotnego driftu - wszystkie kolumny mają PSI < 0.1